# 🔍 Clase 2: Chat con tus Datos (RAG Básico)

## Bienvenido a la Clase 2 - Semana 1

En esta clase aprenderás:
- ✅ Qué son los embeddings y cómo funcionan
- ✅ Búsqueda semántica vs búsqueda por palabras clave
- ✅ Chunking: estrategias para dividir documentos
- ✅ RAG: Retrieval Augmented Generation
- ✅ Implementar un sistema RAG completo
- ✅ Casos de uso: ¿Cuándo usar RAG?
- ✅ Instalación de LLAMA local (opcional)

---

## 📦 Instalación y Configuración

In [ ]:
# Instalación de dependencias
# Paquetes base (todos los necesitan)
!pip install langchain langchain-community chromadb sentence-transformers tiktoken python-dotenv -q

# Instala SOLO el paquete de tu proveedor de LLM:
!pip install langchain-openai -q       # 👈 Si usas OpenAI
!pip install langchain-anthropic -q    # 👈 Si usas Anthropic Claude
!pip install langchain-cohere -q       # 👈 Si usas Cohere

# Para Ollama local (sin API key): instala desde https://ollama.com y ejecuta: ollama run llama3.2

print("✅ Dependencias instaladas correctamente")

In [ ]:
# Imports necesarios
import os
from dotenv import load_dotenv
import numpy as np
from typing import List

# LangChain core (no requieren API key)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Embeddings locales con HuggingFace (gratuito, no requiere API key)
from langchain_community.embeddings import HuggingFaceEmbeddings

# Cargar variables de entorno (.env)
load_dotenv()

print("✅ Librerías importadas correctamente")

In [ ]:
# ============================================================
# ⚙️  CONFIGURACIÓN — Elige tu proveedor de LLM
# ============================================================
# Opciones: "openai" | "anthropic" | "cohere" | "local"
LLM_PROVIDER = "local"   # 👈 Usando Ollama local (tinyllama)

# Si tus API keys NO están en un archivo .env, descomenta la línea correspondiente:
# os.environ["OPENAI_API_KEY"]    = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["COHERE_API_KEY"]    = "..."

# ------------------------------------------------------------

def crear_llm(provider: str = LLM_PROVIDER, temperature: float = 0.7):
    """
    Crea un LLM según el proveedor configurado.
    Soporta: OpenAI, Anthropic, Cohere y Ollama local.
    """
    provider = provider.lower().strip()

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise EnvironmentError("❌ OPENAI_API_KEY no encontrada. Agrégala al .env o configúrala arriba.")
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=temperature, api_key=api_key)
        print("✅ LLM listo → OpenAI GPT-4o-mini")

    elif provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        api_key = os.getenv("ANTHROPIC_API_KEY")
        if not api_key:
            raise EnvironmentError("❌ ANTHROPIC_API_KEY no encontrada. Agrégala al .env o configúrala arriba.")
        llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=temperature, api_key=api_key)
        print("✅ LLM listo → Anthropic Claude Haiku")

    elif provider == "cohere":
        from langchain_cohere import ChatCohere
        api_key = os.getenv("COHERE_API_KEY")
        if not api_key:
            raise EnvironmentError("❌ COHERE_API_KEY no encontrada. Agrégala al .env o configúrala arriba.")
        llm = ChatCohere(model="command-r", temperature=temperature, cohere_api_key=api_key)
        print("✅ LLM listo → Cohere Command-R")

    elif provider == "local":
        from langchain_ollama import OllamaLLM
        # tinyllama (~637MB) funciona bien con el modelo de embeddings en este entorno
        llm = OllamaLLM(model="tinyllama", temperature=temperature)
        print("✅ LLM listo → Ollama local (tinyllama)")

    else:
        raise ValueError(
            f"❌ Proveedor '{provider}' no reconocido.\n"
            "Opciones válidas: 'openai', 'anthropic', 'cohere', 'local'"
        )

    return llm


print(f"⚙️  Configuración cargada | Proveedor activo: {LLM_PROVIDER}")
print("💡 Para cambiar de proveedor, edita LLM_PROVIDER arriba y vuelve a ejecutar esta celda.")

---

## 🧮 Parte 1: ¿Qué son los Embeddings?

### Concepto Fundamental

Los **embeddings** son representaciones numéricas (vectores) de texto que capturan su significado semántico.

**Analogía**: Imagina que cada palabra o frase es un punto en un espacio multidimensional. Palabras con significados similares están cerca unas de otras.

```
"perro" → [0.2, 0.8, 0.1, ...] (vector de 384 dimensiones)
"gato"  → [0.3, 0.7, 0.2, ...] (similar a "perro")
"auto"  → [0.9, 0.1, 0.8, ...] (muy diferente)
```

### ¿Por qué son importantes?

- Permiten comparar textos por **significado**, no solo por palabras
- Fundamentales para búsqueda semántica
- Base de los sistemas RAG

In [ ]:
# Inicializar modelo de embeddings (multilingüe, incluye español)
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("✅ Modelo de embeddings cargado")
print(f"Dimensiones del vector: {len(embeddings_model.embed_query('test'))}")

In [ ]:
# Crear embeddings de ejemplo
textos_ejemplo = [
    "El perro corre en el parque",
    "Un gato juega en el jardín",
    "El automóvil está en el garaje",
    "Los caninos son animales leales"
]

# Generar embeddings
embeddings = [embeddings_model.embed_query(texto) for texto in textos_ejemplo]

print("📊 Embeddings generados:")
for i, texto in enumerate(textos_ejemplo):
    print(f"{i+1}. '{texto}'")
    print(f"   Vector (primeros 5 valores): {embeddings[i][:5]}")
    print()

### Calculando Similitud

Usamos **similitud coseno** para medir qué tan similares son dos embeddings:

In [ ]:
from numpy import dot
from numpy.linalg import norm

def similitud_coseno(vec1, vec2):
    """Calcula la similitud coseno entre dos vectores."""
    return dot(vec1, vec2) / (norm(vec1) * norm(vec2))

# Comparar similitudes
print("🔍 Análisis de Similitud:\n")

comparaciones = [
    (0, 1, "Perro vs Gato"),
    (0, 2, "Perro vs Automóvil"),
    (0, 3, "Perro vs Caninos"),
    (1, 2, "Gato vs Automóvil")
]

for idx1, idx2, descripcion in comparaciones:
    sim = similitud_coseno(embeddings[idx1], embeddings[idx2])
    print(f"{descripcion}:")
    print(f"  '{textos_ejemplo[idx1]}'")
    print(f"  '{textos_ejemplo[idx2]}'")
    print(f"  Similitud: {sim:.4f} {'🟢 Alta' if sim > 0.5 else '🔴 Baja'}\n")

### 💡 Ejercicio 1: Experimenta con Embeddings

In [ ]:
# 👉 Personaliza estos textos
mis_textos = [
    "Escribe tu primer texto aquí",
    "Escribe tu segundo texto aquí",
    "Escribe tu tercer texto aquí"
]

# Generar embeddings
mis_embeddings = [embeddings_model.embed_query(t) for t in mis_textos]

# Comparar
print("Similitud entre texto 1 y 2:", similitud_coseno(mis_embeddings[0], mis_embeddings[1]))
print("Similitud entre texto 1 y 3:", similitud_coseno(mis_embeddings[0], mis_embeddings[2]))
print("Similitud entre texto 2 y 3:", similitud_coseno(mis_embeddings[1], mis_embeddings[2]))

---

## 🔎 Parte 2: Búsqueda Semántica vs Búsqueda por Palabras Clave

### Diferencias Clave

| Aspecto | Búsqueda por Palabras Clave | Búsqueda Semántica |
|---------|----------------------------|--------------------|
| **Método** | Coincidencia exacta de términos | Similitud de significado |
| **Sinónimos** | ❌ No los reconoce | ✅ Los entiende |
| **Contexto** | ❌ Ignora el contexto | ✅ Considera el contexto |
| **Ejemplo** | "auto" ≠ "coche" | "auto" ≈ "coche" |

### Demostración

In [ ]:
# Base de conocimiento de ejemplo
documentos = [
    "Python es un lenguaje de programación versátil y fácil de aprender.",
    "JavaScript se utiliza principalmente para desarrollo web frontend.",
    "La inteligencia artificial está revolucionando la industria tecnológica.",
    "Machine learning es una rama de la IA que permite a las máquinas aprender.",
    "Los algoritmos de deep learning usan redes neuronales profundas.",
    "React es una biblioteca de JavaScript para construir interfaces de usuario."
]

# Generar embeddings de todos los documentos
doc_embeddings = [embeddings_model.embed_query(doc) for doc in documentos]

print(f"✅ {len(documentos)} documentos procesados")

In [ ]:
def busqueda_semantica(query: str, top_k: int = 3):
    """Realiza búsqueda semántica."""
    # Generar embedding de la query
    query_embedding = embeddings_model.embed_query(query)
    
    # Calcular similitudes
    similitudes = [
        (i, similitud_coseno(query_embedding, doc_emb)) 
        for i, doc_emb in enumerate(doc_embeddings)
    ]
    
    # Ordenar por similitud
    similitudes.sort(key=lambda x: x[1], reverse=True)
    
    # Retornar top_k resultados
    return similitudes[:top_k]

# Probar búsqueda semántica
query = "¿Qué es el aprendizaje automático?"

print(f"🔍 Query: '{query}'\n")
print("📋 Resultados de Búsqueda Semántica:\n")

resultados = busqueda_semantica(query)
for rank, (idx, score) in enumerate(resultados, 1):
    print(f"{rank}. Similitud: {score:.4f}")
    print(f"   {documentos[idx]}\n")

**Observa**: Aunque la query usa "aprendizaje automático" y el documento dice "machine learning", ¡la búsqueda semántica los encuentra relacionados!

### 💡 Ejercicio 2: Compara Búsquedas

In [ ]:
# 👉 Prueba diferentes queries
mis_queries = [
    "¿Cómo funcionan las redes neuronales?",
    "Lenguajes para desarrollo web",
    "Frameworks de JavaScript"
]

for query in mis_queries:
    print(f"\n🔍 Query: '{query}'")
    print("="*80)
    resultados = busqueda_semantica(query, top_k=2)
    for rank, (idx, score) in enumerate(resultados, 1):
        print(f"{rank}. [{score:.3f}] {documentos[idx]}")

---

## ✂️ Parte 3: Chunking - Dividir Documentos

### ¿Por qué Chunking?

Los documentos largos necesitan dividirse en fragmentos (chunks) porque:
- Los LLMs tienen límites de tokens (context window)
- Chunks más pequeños = búsqueda más precisa
- Mejor rendimiento y costos

### Estrategias de Chunking

1. **Por caracteres**: Dividir cada N caracteres
2. **Por tokens**: Dividir cada N tokens
3. **Por párrafos**: Usar saltos de línea
4. **Recursivo**: Intentar divisiones naturales (párrafos → oraciones → palabras)

### RecursiveCharacterTextSplitter

La estrategia más común y efectiva:

In [ ]:
# Documento largo de ejemplo
documento_largo = """La inteligencia artificial (IA) es una rama de la informática que se centra en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana.

El machine learning es un subcampo de la IA que permite a las computadoras aprender de los datos sin ser programadas explícitamente. Los algoritmos de ML pueden identificar patrones y hacer predicciones basadas en datos históricos.

El deep learning, a su vez, es una técnica de machine learning que utiliza redes neuronales artificiales con múltiples capas. Estas redes profundas pueden aprender representaciones jerárquicas de los datos, lo que las hace especialmente efectivas para tareas como reconocimiento de imágenes y procesamiento de lenguaje natural.

Los Large Language Models (LLMs) son un tipo de modelo de deep learning entrenado con enormes cantidades de texto. Modelos como GPT-4 han demostrado capacidades impresionantes en comprensión y generación de lenguaje natural."""

print(f"📄 Documento original:")
print(f"Longitud: {len(documento_largo)} caracteres")
print(f"Párrafos: {documento_largo.count(chr(10) + chr(10)) + 1}")

In [ ]:
# Configurar text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,        # Tamaño máximo del chunk
    chunk_overlap=50,      # Overlap entre chunks (para mantener contexto)
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # Prioridad de separadores
)

# Dividir documento
chunks = text_splitter.split_text(documento_largo)

print(f"✂️ Documento dividido en {len(chunks)} chunks:\n")
print("="*80)

for i, chunk in enumerate(chunks, 1):
    print(f"\nChunk {i} ({len(chunk)} caracteres):")
    print("-"*80)
    print(chunk)
    print("="*80)

### Parámetros Importantes

- **chunk_size**: Tamaño ideal del chunk (200-500 tokens típicamente)
- **chunk_overlap**: Overlap para mantener contexto entre chunks
- **separators**: Orden de preferencia para dividir

### 💡 Ejercicio 3: Experimenta con Chunking

In [ ]:
# 👉 Prueba diferentes configuraciones
configuraciones = [
    {"chunk_size": 100, "chunk_overlap": 20},
    {"chunk_size": 300, "chunk_overlap": 50},
    {"chunk_size": 500, "chunk_overlap": 100}
]

for config in configuraciones:
    splitter = RecursiveCharacterTextSplitter(**config)
    chunks = splitter.split_text(documento_largo)
    print(f"Config: {config}")
    print(f"  → Chunks generados: {len(chunks)}")
    print(f"  → Tamaño promedio: {sum(len(c) for c in chunks) / len(chunks):.0f} caracteres\n")

---

## 🔄 Parte 4: RAG - Retrieval Augmented Generation

### ¿Qué es RAG?

RAG combina:
1. **Retrieval (Recuperación)**: Buscar información relevante
2. **Augmentation (Aumentación)**: Agregar información al prompt
3. **Generation (Generación)**: LLM genera respuesta con contexto

### Pipeline RAG

```
Usuario → Query → Embedding → Búsqueda Vectorial → Top K Documentos
                                                            ↓
Usuario ← Respuesta ← LLM ← Prompt + Contexto ← Documentos Recuperados
```

### Implementación Completa

In [ ]:
# Paso 1: Cargar documentos (usaremos el archivo de ejemplo)
try:
    with open('../data/sample_documents.txt', 'r', encoding='utf-8') as f:
        contenido = f.read()
    print("✅ Documentos cargados desde archivo")
except:
    # Si no existe el archivo, usar texto de ejemplo
    contenido = documento_largo
    print("⚠️ Usando documento de ejemplo")

print(f"📊 Tamaño del contenido: {len(contenido)} caracteres")

In [ ]:
# Paso 2: Dividir en chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_text(contenido)
print(f"✂️ Documento dividido en {len(chunks)} chunks")

# Crear objetos Document de LangChain
documents = [Document(page_content=chunk, metadata={"chunk_id": i}) for i, chunk in enumerate(chunks)]
print(f"📄 {len(documents)} documentos creados")

In [ ]:
# Paso 3: Crear base de datos vectorial (ChromaDB)
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings_model,
    collection_name="mi_primera_rag"
)

print("✅ Base de datos vectorial creada")
print(f"📊 Total de vectores: {vectorstore._collection.count()}")

In [ ]:
# Paso 4: Crear LLM y cadena RAG con LCEL (LangChain Expression Language)
llm = crear_llm()

# Crear retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Prompt RAG
prompt_rag = ChatPromptTemplate.from_template("""Responde la pregunta usando únicamente el siguiente contexto.
Si la información no está en el contexto, di "No tengo información sobre eso."

Contexto:
{context}

Pregunta: {question}

Respuesta:""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Cadena RAG usando LCEL
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

print("✅ Chain RAG creado (LCEL)")

In [ ]:
# (Esta celda era una versión alternativa con OpenAI — ya no es necesaria)
# El chain RAG fue creado en la celda anterior usando tu proveedor configurado.
print("ℹ️  Usa la celda anterior para crear el chain RAG con tu proveedor.")

In [ ]:
# Paso 5: ¡Usar el sistema RAG!
def hacer_pregunta(pregunta: str):
    """Hace una pregunta al sistema RAG."""
    print(f"\n{'='*80}")
    print(f"❓ Pregunta: {pregunta}")
    print("="*80)

    respuesta = qa_chain.invoke(pregunta)

    print(f"\n🤖 Respuesta:")
    print("-"*80)
    print(respuesta)
    print("-"*80)

    # Mostrar fuentes
    docs_fuente = retriever.invoke(pregunta)
    print(f"\n📚 Fuentes utilizadas:")
    for i, doc in enumerate(docs_fuente, 1):
        print(f"\n{i}. {doc.page_content[:150]}...")

    return respuesta

# Probar con preguntas
preguntas = [
    "¿Qué es RAG y cómo funciona?",
    "¿Cuál es la diferencia entre machine learning y deep learning?",
    "¿Qué son los embeddings?"
]

for pregunta in preguntas:
    hacer_pregunta(pregunta)

### 💡 Ejercicio 4: Crea tu Propio RAG

In [ ]:
# 👉 Agrega tus propios documentos
mis_documentos = """
Escribe aquí tu propio contenido.
Puede ser sobre cualquier tema que te interese.
Cuanto más contenido agregues, mejor funcionará el RAG.

Puedes incluir múltiples párrafos.
El sistema los dividirá automáticamente en chunks.
"""

# Crear RAG con tus documentos
mis_chunks = text_splitter.split_text(mis_documentos)
mis_docs = [Document(page_content=chunk) for chunk in mis_chunks]

mi_vectorstore = Chroma.from_documents(
    documents=mis_docs,
    embedding=embeddings_model,
    collection_name="mi_rag_personalizado"
)

mi_retriever = mi_vectorstore.as_retriever()

mi_qa_chain = (
    {"context": mi_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

# 👉 Haz tu pregunta
mi_pregunta = "Escribe tu pregunta aquí"
print(f"\n❓ {mi_pregunta}")
print(mi_qa_chain.invoke(mi_pregunta))

---

## 🎯 Parte 5: ¿Cuándo Usar RAG?

### ✅ Casos de Uso Ideales para RAG

1. **Información actualizada**
   - Noticias, precios, eventos recientes
   - Datos que cambian frecuentemente

2. **Documentación interna**
   - Manuales de empresa
   - Políticas y procedimientos
   - Base de conocimiento técnica

3. **Dominios especializados**
   - Legal, médico, técnico
   - Información muy específica

4. **Necesidad de citas**
   - Cuando necesitas mostrar fuentes
   - Verificabilidad importante

5. **Grandes volúmenes de datos**
   - Bibliotecas de documentos
   - Repositorios de conocimiento

### ❌ Cuándo NO Usar RAG

1. **Conocimiento general**
   - El LLM ya lo sabe
   - No necesitas fuentes específicas

2. **Tareas creativas**
   - Escritura creativa
   - Brainstorming
   - Generación de ideas

3. **Conversación casual**
   - Chat general
   - No requiere información específica

4. **Datos muy pequeños**
   - Si cabe en el prompt, úsalo directamente
   - RAG agrega complejidad innecesaria

### Comparación

| Escenario | Sin RAG | Con RAG |
|-----------|---------|----------|
| "¿Qué es Python?" | ✅ Perfecto | ❌ Innecesario |
| "¿Cuál es nuestra política de vacaciones?" | ❌ No lo sabe | ✅ Perfecto |
| "Escribe un poema" | ✅ Perfecto | ❌ Innecesario |
| "Resume este documento de 100 páginas" | ❌ No cabe | ✅ Perfecto |

---

## 🦙 Parte 6: LLAMA Local (Opcional)

### ¿Por qué LLAMA Local?

**Ventajas**:
- ✅ Privacidad total (datos no salen de tu máquina)
- ✅ Sin costos de API
- ✅ Sin límites de rate
- ✅ Funciona offline

**Desventajas**:
- ❌ Requiere recursos (RAM, GPU opcional)
- ❌ Modelos más pequeños = menor calidad
- ❌ Más lento que APIs

### Requisitos del Sistema

- **RAM**: Mínimo 8GB (16GB recomendado)
- **Espacio**: 5-10GB para modelos
- **GPU**: Opcional pero mejora velocidad significativamente

### Instalación

In [ ]:
# Instalación de llama-cpp-python
# Para CPU:
# !pip install llama-cpp-python

# Para GPU (NVIDIA):
# !CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install llama-cpp-python

print("⚠️ Descomenta las líneas de arriba para instalar")

### Descargar Modelo

Modelos recomendados de Hugging Face:

1. **Pequeño (2-3GB)**: `TheBloke/Llama-2-7B-Chat-GGUF`
2. **Mediano (5-7GB)**: `TheBloke/Llama-2-13B-Chat-GGUF`
3. **Grande (10GB+)**: `TheBloke/Llama-2-70B-Chat-GGUF`

**Pasos**:
1. Ir a https://huggingface.co/TheBloke
2. Buscar modelo GGUF
3. Descargar archivo `.gguf`
4. Guardar en carpeta `models/`

In [ ]:
# Ejemplo de uso con LLAMA local (requiere modelo descargado)
# from langchain_community.llms import LlamaCpp

# llama_local = LlamaCpp(
#     model_path="./models/llama-2-7b-chat.Q4_K_M.gguf",
#     temperature=0.7,
#     max_tokens=2000,
#     n_ctx=2048,  # Context window
#     verbose=True
# )

# # Usar con RAG
# qa_chain_local = RetrievalQA.from_chain_type(
#     llm=llama_local,
#     retriever=vectorstore.as_retriever(),
#     return_source_documents=True
# )

print("💡 Código de ejemplo para LLAMA local (comentado)")
print("Descomenta y ajusta la ruta del modelo para usar")

---

## 📊 Parte 7: Mejores Prácticas de RAG

### 1. Tamaño de Chunks

```python
# ❌ Muy pequeño (pierde contexto)
chunk_size=50

# ✅ Balanceado
chunk_size=500

# ⚠️ Muy grande (menos preciso)
chunk_size=2000
```

### 2. Overlap

```python
# Mantiene contexto entre chunks
chunk_overlap=50  # 10% del chunk_size es buena práctica
```

### 3. Número de Documentos (k)

```python
# ❌ Muy pocos (puede perder info relevante)
k=1

# ✅ Balanceado
k=3

# ⚠️ Muchos (ruido, costo, límite de tokens)
k=10
```

### 4. Metadata

Agrega metadata útil a tus documentos:

In [ ]:
# Ejemplo con metadata
docs_con_metadata = [
    Document(
        page_content="Contenido del documento",
        metadata={
            "source": "manual.pdf",
            "page": 5,
            "author": "Juan Pérez",
            "date": "2024-01-15",
            "category": "técnico"
        }
    )
]

print("✅ Metadata ayuda a filtrar y citar fuentes")

### 5. Filtrado por Metadata

In [ ]:
# Buscar solo en documentos de cierta categoría
# retriever = vectorstore.as_retriever(
#     search_kwargs={
#         "k": 3,
#         "filter": {"category": "técnico"}
#     }
# )

print("💡 Puedes filtrar por metadata para búsquedas más precisas")

---

## 🎓 Resumen de la Clase

### Conceptos Aprendidos

1. ✅ **Embeddings**: Vectores que representan significado semántico
2. ✅ **Similitud Coseno**: Medir similitud entre embeddings
3. ✅ **Búsqueda Semántica**: Buscar por significado, no palabras
4. ✅ **Chunking**: Dividir documentos en fragmentos manejables
5. ✅ **RAG**: Retrieval + Augmentation + Generation
6. ✅ **ChromaDB**: Base de datos vectorial
7. ✅ **LangChain**: Framework para RAG
8. ✅ **LLAMA Local**: Alternativa privada y offline

### Pipeline RAG Completo

```python
# 1. Cargar documentos
documentos = cargar_documentos()

# 2. Dividir en chunks
chunks = text_splitter.split_text(documentos)

# 3. Crear embeddings y vectorstore
vectorstore = Chroma.from_documents(chunks, embeddings)

# 4. Crear retriever
retriever = vectorstore.as_retriever()

# 5. Crear chain RAG
qa_chain = RetrievalQA.from_chain_type(llm, retriever=retriever)

# 6. ¡Usar!
respuesta = qa_chain.invoke({"query": "tu pregunta"})
```

### Mejores Prácticas

- 📏 Chunk size: 200-500 tokens
- 🔄 Overlap: 10% del chunk size
- 📊 Top k: 3-5 documentos
- 🏷️ Usa metadata para filtrar
- 🎯 Evalúa la calidad de las respuestas

### Próximos Pasos

En la **Semana 2** aprenderemos:
- 🔗 LangChain en profundidad
- 🤖 Crear agentes inteligentes
- 🛠️ Herramientas (tools) para agentes
- 🌐 Integración con APIs

---

## 📚 Recursos Adicionales

- [LangChain RAG Tutorial](https://python.langchain.com/docs/use_cases/question_answering/)
- [ChromaDB Documentation](https://docs.trychroma.com/)
- [Sentence Transformers](https://www.sbert.net/)
- [LLAMA Models](https://huggingface.co/TheBloke)

---

## 🎯 Tarea para Casa

1. Crea un sistema RAG con tus propios documentos (PDFs, artículos, etc.)
2. Experimenta con diferentes tamaños de chunks
3. Compara búsqueda semántica vs keyword search
4. Usa el script `db_migration.py` para migrar tus datos
5. (Opcional) Instala LLAMA local y pruébalo

**¡Nos vemos en la Semana 2! 🚀**